# Trace-Bench M2 — Quick Validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/guru-code-expert/Trace-Bench/blob/m2/coverage/notebooks/03_m2_quick_validation.ipynb)

Lightweight smoke test covering the **core M2 contracts** without blowing Colab RAM:

| Check | What it proves | Jobs |
|-------|---------------|------|
| Parallel run | max_workers=2 produces same results as sequential | 7 |
| Resume | Re-run reuses completed jobs (manifest `status: reused`) | 7 |
| Stub coverage | 5 LLM4AD + 2 internal tasks load, train, eval | 7 |
| Real-mode (opt.) | Same 5 LLM4AD tasks with real LLM if API key present | 5 |

**Runtime:** ~3 min stub, ~10 min real on free Colab.

For full 65-task coverage, use **`02_m2_coverage.ipynb`**.

In [ ]:
# Mount Drive (optional) + compute persistent runs_dir + detect API key
from datetime import date
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass


def bench_dir(project="bench", sub="trace_bench_m2_quick", local="/content/bench"):
    drive_root = Path("/content/drive/MyDrive")
    root = drive_root if drive_root.is_dir() else Path(local)
    out = root / project / date.today().isoformat() / sub
    out.mkdir(parents=True, exist_ok=True)
    return str(out)

RUNS_DIR = bench_dir()
os.environ["RUNS_DIR"] = RUNS_DIR
print("Runs dir:", RUNS_DIR)

# --- Auto-detect API key ---
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("OPENROUTER_API_KEY") or ""
    except Exception:
        pass

MODEL = os.environ.get("OPENROUTER_MODEL", "openrouter/x-ai/grok-4.1-fast")

if API_KEY:
    os.environ["OPENROUTER_API_KEY"] = API_KEY
    os.environ["OPENAI_API_KEY"] = API_KEY
    os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"
    os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
    os.environ["TRACE_DEFAULT_LLM_BACKEND"] = "LiteLLM"
    os.environ["TRACE_LITELLM_MODEL"] = MODEL
    MODE = "real"
    print(f"API key found - running in REAL mode (model: {MODEL})")
else:
    MODE = "stub"
    print("No OPENROUTER_API_KEY found. Running in STUB mode.")

os.environ["TB_MODE"] = MODE
print(f"Mode: {MODE}")

In [ ]:
# Clone repos + install deps
!git clone --depth 1 --branch m2/coverage https://github.com/guru-code-expert/Trace-Bench.git 2>/dev/null || (cd Trace-Bench && git pull)
!git clone --depth 1 --branch experimental https://github.com/guru-code-expert/OpenTrace.git 2>/dev/null || (cd OpenTrace && git pull)

%cd Trace-Bench

!apt-get update -qq && apt-get install -y -qq graphviz > /dev/null 2>&1
!python -m pip install -q pyyaml pytest numpy matplotlib graphviz litellm==1.75.0 \
    "aiohttp>=3.9,<3.13" scipy networkx gymnasium gym pandas datasets sympy pymoo

## 1. Stub-Mode Run (7 jobs, parallel)

Run 5 LLM4AD + 2 internal tasks with PrioritySearch, `max_workers=2`.

In [ ]:
%%bash
set -euo pipefail
cd /content/Trace-Bench

echo "=== Stub-mode quick run (7 tasks, max_workers=2) ==="

PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run \
    --config configs/m2_quick.yaml \
    --runs-dir "$RUNS_DIR/stub" \
    --max-workers 2

echo ""
echo "Stub run complete."

In [ ]:
# Verify stub results
import json, pathlib, pandas as pd

def latest_run(root):
    root = pathlib.Path(root)
    if not root.exists():
        return None
    candidates = sorted([p for p in root.iterdir() if p.is_dir()],
                        key=lambda p: p.stat().st_mtime)
    for p in reversed(candidates):
        if (p / "results.csv").exists():
            return p
    return None

stub_dir = latest_run(f"{RUNS_DIR}/stub")
assert stub_dir, "No stub run found!"
print(f"Run: {stub_dir.name}")

df = pd.read_csv(stub_dir / "results.csv")
print(f"Total jobs: {len(df)}")
print(f"\nStatus breakdown:")
print(df["status"].value_counts())

ok_count = len(df[df["status"] == "ok"])
print(f"\nOK: {ok_count}/{len(df)}")

print(f"\n{'Task':<55} {'Status':<8} {'Score'}")
print("-" * 80)
for _, row in df.sort_values("task_id").iterrows():
    print(f"  {row['task_id']:<53} {row['status']:<8} {row.get('score_best', 'N/A')}")

assert ok_count >= 5, f"Expected >=5 OK jobs, got {ok_count}"
print(f"\nPASS: {ok_count} jobs OK")

## 2. Resume Check

Re-run the exact same config with `resume: auto`.
Previously-OK jobs should be reused; previously-failed jobs get re-attempted.

In [ ]:
# Resume run: re-run same config with the SAME run_id so existing jobs are found
import sys, os, json, pathlib

sys.path.insert(0, "/content/OpenTrace")
sys.path.insert(0, "/content/Trace-Bench")
os.chdir("/content/Trace-Bench")

from trace_bench.config import RunConfig
from trace_bench.runner import BenchRunner

# Find the first run's run_id
stub_root = pathlib.Path(f"{RUNS_DIR}/stub")
first_run = sorted([p for p in stub_root.iterdir() if p.is_dir()],
                   key=lambda p: p.stat().st_mtime)[-1]
run_id = first_run.name
print(f"Resuming into existing run: {run_id}")

cfg = RunConfig.from_dict({
    "tasks": [
        {"id": "internal:numeric_param"},
        {"id": "internal:code_param"},
        {"id": "llm4ad:circle_packing"},
        {"id": "llm4ad:optimization/knapsack_construct"},
        {"id": "llm4ad:optimization/bp_1d_construct"},
        {"id": "llm4ad:science_discovery/oscillator1"},
        {"id": "llm4ad:machine_learning/acrobot"},
    ],
    "trainers": [{"id": "PrioritySearch", "params_variants": [
        {"ps_steps": 1, "ps_batches": 1, "ps_candidates": 2, "ps_proposals": 2}
    ]}],
    "seeds": [123],
    "mode": "stub",
    "resume": "auto",
})
cfg.runs_dir = str(stub_root)
cfg.run_id = run_id

runner = BenchRunner(cfg)
summary = runner.run()
print(f"Resume run complete. {len(summary.results)} results.")

In [ ]:
# Verify resume semantics: OK jobs reused, failed jobs re-attempted
import json, pathlib

stub_root = pathlib.Path(f"{RUNS_DIR}/stub")
runs = sorted([p for p in stub_root.iterdir() if p.is_dir()],
              key=lambda p: p.stat().st_mtime)
resume_dir = runs[-1]
manifest = json.loads((resume_dir / "meta" / "manifest.json").read_text())

statuses = {j["task_id"]: j["status"] for j in manifest["jobs"]}
reused = sum(1 for s in statuses.values() if s == "reused")
retried = sum(1 for s in statuses.values() if s != "reused")

print(f"Resume run: {resume_dir.name}")
print(f"Reused: {reused}, Re-attempted: {retried}")
for j in manifest["jobs"]:
    print(f"  {j['task_id']:<55} {j['status']}")

# resume=auto must reuse OK jobs, re-run failed ones.
# At least the OK jobs from run 1 should be reused.
assert reused >= 5, \
    f"Expected >=5 reused (OK jobs from first run), got {reused}"
print(f"\nPASS: {reused} jobs reused, {retried} re-attempted -- resume works correctly.")

## 3. Artifact Integrity

Verify the run directory has the expected M2 layout.

In [ ]:
import json, pathlib

stub_dir = latest_run(f"{RUNS_DIR}/stub")

# Required artifacts
expected = [
    "results.csv",
    "summary.json",
    "meta/manifest.json",
    "meta/config.snapshot.yaml",
    "meta/env.json",
]

print(f"Run: {stub_dir.name}")
print(f"\nArtifact check:")
all_ok = True
for rel in expected:
    p = stub_dir / rel
    exists = p.exists()
    size = p.stat().st_size if exists else 0
    mark = "OK" if exists and size > 0 else "MISSING"
    print(f"  [{mark}] {rel}  ({size:,} bytes)")
    if not exists or size == 0:
        all_ok = False

# Check per-job artifacts
jobs_dir = stub_dir / "jobs"
job_dirs = list(jobs_dir.iterdir()) if jobs_dir.exists() else []
print(f"\nJob directories: {len(job_dirs)}")
for jd in sorted(job_dirs)[:3]:  # show first 3
    has_results = (jd / "results.json").exists()
    has_meta = (jd / "job_meta.json").exists()
    print(f"  {jd.name}  results.json={'OK' if has_results else 'MISS'}  job_meta.json={'OK' if has_meta else 'MISS'}")
if len(job_dirs) > 3:
    print(f"  ... and {len(job_dirs) - 3} more")

# Verify summary.json content
summary = json.loads((stub_dir / "summary.json").read_text())
print(f"\nSummary:")
print(f"  total_jobs:  {summary.get('total_jobs')}")
print(f"  ok:          {summary.get('ok')}")
print(f"  failed:      {summary.get('failed')}")
print(f"  skipped:     {summary.get('skipped')}")

assert all_ok, "Some artifacts are missing!"
print("\nPASS: All artifacts present and non-empty.")

## 4. Real-Mode Quick Check (API Key Required)

Run the same 5 LLM4AD tasks with a real LLM backend (1 training step).
Skipped automatically if no `OPENROUTER_API_KEY` is set.

In [ ]:
import sys, os, json

if os.environ.get("TB_MODE") != "real":
    print("SKIP: Set OPENROUTER_API_KEY to run real-mode check.")
else:
    from pathlib import Path

    config = {
        "mode": "real",
        "seeds": [123],
        "max_workers": 2,
        "resume": "auto",
        "job_timeout": 300,
        "trainers": [
            {
                "id": "PrioritySearch",
                "params_variants": [
                    {
                        "ps_steps": 1,
                        "ps_batches": 1,
                        "ps_candidates": 2,
                        "ps_proposals": 2,
                    }
                ],
            }
        ],
        "tasks": [
            {"id": "llm4ad:circle_packing"},
            {"id": "llm4ad:optimization/knapsack_construct"},
            {"id": "llm4ad:optimization/bp_1d_construct"},
            {"id": "llm4ad:science_discovery/oscillator1"},
            {"id": "llm4ad:machine_learning/acrobot"},
        ],
    }

    config_path = "/content/m2_quick_real.json"
    Path(config_path).write_text(json.dumps(config, indent=2))
    print(f"Real-mode config: 5 tasks, ps_steps=1, max_workers=2")

    import subprocess
    env = dict(os.environ)
    env["PYTHONPATH"] = "/content/OpenTrace:" + env.get("PYTHONPATH", "")
    result = subprocess.run(
        [
            sys.executable, "-m", "trace_bench", "run",
            "--config", config_path,
            "--runs-dir", f"{os.environ['RUNS_DIR']}/real",
        ],
        cwd="/content/Trace-Bench",
        env=env,
        capture_output=True,
        text=True,
    )
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode != 0:
        print(f"STDERR (last 1000 chars):\n{result.stderr[-1000:]}")
    print("\nReal-mode quick check complete.")

In [ ]:
# Analyze real-mode results
import os

if os.environ.get("TB_MODE") != "real":
    print("SKIP: Real-mode not active.")
else:
    import pandas as pd, pathlib

    real_dir = latest_run(f"{RUNS_DIR}/real")
    if real_dir:
        df_real = pd.read_csv(real_dir / "results.csv")
        print(f"Real-mode run: {real_dir.name}")
        print(f"Total jobs: {len(df_real)}")
        print(f"\nStatus: {dict(df_real['status'].value_counts())}")

        print(f"\n{'Task':<55} {'Status':<8} {'Initial':>8} {'Best':>8}")
        print("-" * 85)
        for _, row in df_real.sort_values("task_id").iterrows():
            si = row.get("score_initial", "")
            sb = row.get("score_best", "")
            print(f"  {row['task_id']:<53} {row['status']:<8} {str(si):>8} {str(sb):>8}")

        ok = len(df_real[df_real["status"] == "ok"])
        pct = ok / len(df_real) * 100
        print(f"\nSuccess: {ok}/{len(df_real)} = {pct:.0f}%")
        if pct >= 60:
            print(f"PASS: Real-mode quick check {pct:.0f}% >= 60%")
        else:
            print(f"BELOW TARGET: {pct:.0f}% < 60%")
    else:
        print("No real-mode runs found.")

## Summary

| Check | Expected |
|-------|----------|
| Stub run | >= 5/7 jobs OK |
| Resume | OK jobs reused, failed jobs re-attempted |
| Artifacts | results.csv, summary.json, meta/manifest.json, per-job dirs |
| Real-mode | >= 3/5 tasks OK (if API key present) |

For full 65-task coverage, 10-step optimization, and fail_fast tests, see **`02_m2_coverage.ipynb`**.